# Lie Groups — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Lie Groups exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/lie_companion.ipynb)

## Setup

In [ ]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
import numpy as np
from numpy import einsum as ein
from typing import List

from symm4ml import groups, linalg, rep, lie

### Reference data

These structure constants and generators are used throughout the exercise for testing.

In [ ]:
# SO(3) structure constants
so3_A = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) L=1 generators
so3_X = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(1,3) structure constants
so13_A = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
        ],
        [
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SO(1,3) generators
so13_X = np.array(
    [
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, -1.0],
            [0.0, 0.0, 1.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, -1.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, -1.0, 0.0],
            [0.0, 1.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
        ],
        [
            [0.0, 0.0, 0.0, 1.0],
            [0.0, 0.0, 0.0, 0.0],
            [0.0, 0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0, 0.0],
        ],
    ]
)

# SU(2) generators
su2_X = np.array(
    [
        [
            [0.0 + 0.0j, 0.5 + 0.0j],
            [-0.5 + 0.0j, 0.0 + 0.0j],
        ],
        [
            [-0.0 - 0.5j, 0.0 + 0.0j],
            [0.0 + 0.0j, 0.0 + 0.5j],
        ],
        [
            [0.0 - 0.0j, 0.0 + 0.5j],
            [0.0 + 0.5j, 0.0 - 0.0j],
        ],
    ]
)

print(f"so3_A: {so3_A.shape}, so3_X: {so3_X.shape}")
print(f"so13_A: {so13_A.shape}, so13_X: {so13_X.shape}")
print(f"su2_X: {su2_X.shape}")

---
## 1. `are_isomorphic(X1, X2)`

Check if two representations of a Lie algebra (expressed in terms of generators) are isomorphic, i.e., the same up to a similarity transform. You can use a function from a previous problem set.

In [ ]:
def are_isomorphic(X1: np.ndarray, X2: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if two representations of a Lie group are isomorphic.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        are_isomorphic: bool - True if the representations are isomorphic.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from lie.py
assert are_isomorphic(so3_X, so3_X)
print("are_isomorphic tests passed!")

---
## 2. `tensor_product(X1, X2)`

Compute the tensor product of two representations of a Lie algebra, where both input and output are expressed in terms of generators.

In [ ]:
def tensor_product(X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
    """Tensor product of two representations of a Lie group.
    Input:
        X1: np.array [lie_dim, d1, d1] - generators of a representation.
        X2: np.array [lie_dim, d2, d2] - generators of a representation.
    Output:
        X: np.array [lie_dim, d1*d2, d1*d2] - tensor product of the representations.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from lie.py
assert tensor_product(so3_X, so3_X).shape == (3, 9, 9)
print("tensor_product tests passed!")

---
## 3. `is_a_representation(algebra, X)`

Check whether the given generators `X` satisfy the commutation relations encoded in `algebra`: $[X_i, X_j] = \sum_k A_{ijk} X_k$.

In [ ]:
def is_a_representation(
    algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8
) -> bool:
    """Check if X satisfies the commutation relations of the Lie algebra:
        [X_i, X_j] = sum_k A_{ijk} X_k
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_a_representation: bool - True if X satisfies the commutation relations.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from lie.py
assert is_a_representation(so3_A, so3_X)
print("is_a_representation tests passed!")

---
## 4. `is_an_irrep(algebra, X)`

Check if a representation of a Lie algebra is irreducible. Return `True` if the input is a valid representation AND is irreducible.

In [ ]:
def is_an_irrep(algebra: np.ndarray, X: np.ndarray, *, tol: float = 1e-8) -> bool:
    """Checks if X is an irreducible representation of the Lie algebra.
    Input:
        algebra: np.array [lie_dim, lie_dim, lie_dim] - Lie algebra (structure constants)
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        is_an_irrep: bool - True if X is an irreducible representation.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from lie.py
assert is_an_irrep(so3_A, so3_X)
print("is_an_irrep tests passed!")

---
## 5. `decompose_rep_into_irreps(X)`

Decompose a representation of a Lie algebra into a direct sum of irreducible representations. Follow the same algorithm as for finite groups.

In [ ]:
def decompose_rep_into_irreps(X: np.array, *, tol: float = 1e-8) -> List[np.array]:
    """Decomposes representation into irreducible representations.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
    Output:
        Ys: List[np.array] - list of generators of irreducible representations.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from lie.py
assert {
    X.shape[1] for X in decompose_rep_into_irreps(tensor_product(so3_X, so3_X))
} == {1, 3, 5}
print("decompose_rep_into_irreps tests passed!")

---
## 6. `infer_irreps_from_tensor_products(X, n)`

Infer `n` non-isomorphic finite-dimensional irreps of the underlying matrix Lie group by successively decomposing tensor products. Start with the trivial representation and take successive tensor products with `X` and (if needed) its conjugate `X*`.

In [ ]:
def infer_irreps_from_tensor_products(
    X: np.ndarray, n: int, *, tol: float = 1e-8
) -> List[np.ndarray]:
    """Infers irreducible representations from successive tensor products of a representation.
    Input:
        X: np.array [lie_dim, d, d] - generators of a representation.
        n: int - number of non-isomorphic irreducible representations to infer.
    Output:
        Ys: List[np.array] - list of n generators of irreducible representations,
            each an np.array of shape [lie_dim, d', d'] for some d'.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from lie.py
Xs = infer_irreps_from_tensor_products(so3_X, 3)
assert len(Xs) == 3
assert Xs[0].shape == (3, 1, 1)
assert is_an_irrep(so3_A, Xs[0])
assert Xs[1].shape == (3, 3, 3)
assert is_an_irrep(so3_A, Xs[1])
assert Xs[2].shape == (3, 5, 5)
assert is_an_irrep(so3_A, Xs[2])
print("infer_irreps_from_tensor_products tests passed!")

---
## Explore Further

In [ ]:
# Try inferring irreps of SO(1,3) or SU(2)!